# Figure out how to hit Kalshi API

In [19]:
import requests, json

url = "https://external-api.kalshi.com/trade-api/v2/series/KXHIGHNY"
response = requests.get(url)
series_data = response.json()

print(f"Series Title: {series_data['series']['title']}")
print(f"Frequency: {series_data['series']['frequency']}")
print(f"Category: {series_data['series']['category']}")

Series Title: Highest temperature in NYC
Frequency: daily
Category: Climate and Weather


In [20]:
# Get all open markets for the KXHIGHNY series
markets_url = f"https://external-api.kalshi.com/trade-api/v2/markets?series_ticker=KXHIGHNY&status=open"
markets_response = requests.get(markets_url)
markets_data = markets_response.json()

print(f"\nActive markets in KXHIGHNY series:")
for market in markets_data['markets']:
    print(f"- {market['ticker']}: {market['title']}")
    print(f"  Event: {market['event_ticker']}")
    print(f"  Yes Price: ${market['yes_bid_dollars']} | Volume: {market['volume_fp']}")
    print()

# Get details for a specific event if you have its ticker
if markets_data['markets']:
    # Let's get details for the first market's event
    event_ticker = markets_data['markets'][0]['event_ticker']
    event_url = f"https://external-api.kalshi.com/trade-api/v2/events/{event_ticker}"
    event_response = requests.get(event_url)
    event_data = event_response.json()

    print(f"Event Details:")
    print(f"Title: {event_data['event']['title']}")
    print(f"Category: {event_data['event']['category']}")


Active markets in KXHIGHNY series:
- KXHIGHNY-26JUN13-T94: Will the **high temp in NYC** be >94° on Jun 13, 2026?
  Event: KXHIGHNY-26JUN13
  Yes Price: $0.0000 | Volume: 6314.16

- KXHIGHNY-26JUN13-T87: Will the **high temp in NYC** be <87° on Jun 13, 2026?
  Event: KXHIGHNY-26JUN13
  Yes Price: $0.2700 | Volume: 5554.03

- KXHIGHNY-26JUN13-B93.5: Will the **high temp in NYC** be 93-94° on Jun 13, 2026?
  Event: KXHIGHNY-26JUN13
  Yes Price: $0.0100 | Volume: 2361.64

- KXHIGHNY-26JUN13-B91.5: Will the **high temp in NYC** be 91-92° on Jun 13, 2026?
  Event: KXHIGHNY-26JUN13
  Yes Price: $0.0500 | Volume: 3010.29

- KXHIGHNY-26JUN13-B89.5: Will the **high temp in NYC** be 89-90° on Jun 13, 2026?
  Event: KXHIGHNY-26JUN13
  Yes Price: $0.1900 | Volume: 3539.61

- KXHIGHNY-26JUN13-B87.5: Will the **high temp in NYC** be 87-88° on Jun 13, 2026?
  Event: KXHIGHNY-26JUN13
  Yes Price: $0.4800 | Volume: 3974.35

- KXHIGHNY-26JUN12-T97: Will the **high temp in NYC** be >97° on Jun 12, 2026?

# Get list of all settled markets for NBA games
 - Must query live and historical endpoints

In [21]:
cutoff_url = f"https://external-api.kalshi.com/trade-api/v2/historical/cutoff"
cutoff_response = requests.get(cutoff_url)
cutoff_data = cutoff_response.json()


print(json.dumps(cutoff_data, indent=4))


{
    "market_settled_ts": "2026-04-13T00:00:00Z",
    "orders_updated_ts": "2026-04-13T00:00:00Z",
    "trades_created_ts": "2026-04-13T00:00:00Z"
}


In [22]:
import requests, json, re, time

BASE = "https://external-api.kalshi.com/trade-api/v2"
PAGE_LIMIT = 1000           # max page size Kalshi allows on list endpoints
NBA_RE = re.compile(r"\bNBA\b")   # \b boundary => matches "NBA" but not "WNBA"
 
session = requests.Session()
session.headers.update({"Accept": "application/json"})
 
 
def _get(path, params=None):
    """GET with simple exponential backoff on 429 / 5xx."""
    url = f"{BASE}{path}"
    delay = 1.0
    for attempt in range(6):
        resp = session.get(url, params=params, timeout=30)
        if resp.status_code == 429 or resp.status_code >= 500:
            time.sleep(delay)
            delay = min(delay * 2, 30)
            continue
        resp.raise_for_status()
        return resp.json()
    resp.raise_for_status()  # surface the last error if we never succeeded
 
 
def paginate(path, params, list_key):
    """
    Yield every item from a cursor-paginated Kalshi list endpoint.
 
    `list_key` is the field holding the array, e.g. "markets" or "series".
    Kalshi returns a top-level "cursor"; an empty/absent cursor means done.
    """
    params = dict(params or {})
    params.setdefault("limit", PAGE_LIMIT)
    while True:
        data = _get(path, params)
        for item in data.get(list_key, []) or []:
            yield item
        cursor = data.get("cursor")
        if not cursor:
            return
        params["cursor"] = cursor
 
 
def discover_nba_series():
    """Return the list of series tickers whose title/tags reference the NBA."""
    tickers = []
    for series in paginate("/series", {"category": "Sports"}, "series"):
        title = series.get("title", "") or ""
        tags = " ".join(series.get("tags", []) or [])
        ticker = series.get("ticker", "") or ""
        # Match on title or tags; fall back to the conventional KXNBA prefix.
        if NBA_RE.search(title) or NBA_RE.search(tags) or ticker.upper().startswith("KXNBA"):
            tickers.append(ticker)
    return sorted(set(tickers))
 
 
def markets_for_series(series_ticker):
    """All settled markets for one series available on the live API (every status: active, closed, settled...)."""
    # Omitting `status` returns markets of any status from the live data set.
    yield from paginate("/markets", {"series_ticker": series_ticker, "status": "settled"}, "markets")
 
 
def historical_markets_for_series(series_ticker):
    """Markets for one series that have been archived past the historical cutoff."""
    yield from paginate(
        "/historical/markets", {"series_ticker": series_ticker}, "markets"
    )
 
 
def pull_all_nba_markets():
    nba_series = discover_nba_series()
    print(f"Found {len(nba_series)} NBA series: {nba_series}")
 
    by_ticker = {}          # market ticker -> market object (dedup)
    source = {}             # market ticker -> {"live", "historical"} for provenance
 
    for s in nba_series:
        for m in markets_for_series(s):
            t = m["ticker"]
            by_ticker[t] = m
            source.setdefault(t, set()).add("live")
 
        for m in historical_markets_for_series(s):
            t = m["ticker"]
            by_ticker.setdefault(t, m)          # don't overwrite the fresher live copy
            source.setdefault(t, set()).add("historical")
 
    markets = list(by_ticker.values())
    print(f"Total unique NBA markets: {len(markets)}")
    print(f"  live-only:       {sum(1 for v in source.values() if v == {'live'})}")
    print(f"  historical-only: {sum(1 for v in source.values() if v == {'historical'})}")
    print(f"  in both:         {sum(1 for v in source.values() if len(v) == 2)}")
 
    return markets, {t: sorted(v) for t, v in source.items()}

In [23]:
markets, source = pull_all_nba_markets()

Found 187 NBA series: ['KXCITYNBAEXPAND', 'KXCOACHOUTNBA', 'KXCOACHOUTNBADATE', 'KXFIRSTPICKNBA', 'KXLEADERNBA3PT', 'KXLEADERNBAAST', 'KXLEADERNBABLK', 'KXLEADERNBAPTS', 'KXLEADERNBAREB', 'KXLEADERNBASTL', 'KXNBA', 'KXNBA1HSPREAD', 'KXNBA1HTOTAL', 'KXNBA1HWINNER', 'KXNBA1QSPREAD', 'KXNBA1QTOTAL', 'KXNBA1QWINNER', 'KXNBA1STTEAM', 'KXNBA1STTEAMDEF', 'KXNBA2D', 'KXNBA2HSPREAD', 'KXNBA2HTOTAL', 'KXNBA2HWINNER', 'KXNBA2NDTEAM', 'KXNBA2NDTEAMDEF', 'KXNBA2QSPREAD', 'KXNBA2QTOTAL', 'KXNBA2QWINNER', 'KXNBA30COMEBACK', 'KXNBA3D', 'KXNBA3PEAT', 'KXNBA3PT', 'KXNBA3PTALLGAMES', 'KXNBA3PTCONTEST', 'KXNBA3QSPREAD', 'KXNBA3QTOTAL', 'KXNBA3QWINNER', 'KXNBA3RDTEAM', 'KXNBA4QSPREAD', 'KXNBA4QTOTAL', 'KXNBA4QWINNER', 'KXNBAALLSTAR', 'KXNBAALLSTARGAME', 'KXNBAALLSTARMVP', 'KXNBAALLSTARS', 'KXNBAAPOLOGY', 'KXNBAAST', 'KXNBAATLANTIC', 'KXNBABLK', 'KXNBACELEBRITY3PT', 'KXNBACELEBRITYGAME', 'KXNBACENTRAL', 'KXNBACLUTCH', 'KXNBACOY', 'KXNBACUP', 'KXNBACUPMVP', 'KXNBADPOY', 'KXNBADRAFT1', 'KXNBADRAFT10', 'KXNBAD

In [24]:
with open("nba_markets.json", "w") as f:
    json.dump(markets, f, indent=4)

with open("nba_source.json", "w") as f:
    json.dump(source, f, indent=4)